# iDILI-Predict: Interactive Walkthrough

This notebook demonstrates the iDILI-Predict pipeline step by step using the included example data (63D cell line). Each stage can be run independently for inspection and customization.

**Pipeline stages:**
1. Load and inspect data
2. Encode labels and impute missing values
3. Normalize features (MAD robustize per plate)
4. Train AutoGluon with compound-grouped cross-validation
5. Evaluate and visualize OOF predictions

## 1. Setup

In [ ]:
import sys
from pathlib import Path

# Add parent directory to path so we can import idili_predict
# (Not needed if you ran `pip install -e .` from the repo root)
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd

from idili_predict.utils import (
    load_column_types,
    validate_required_columns,
    filter_to_present_numeric,
    encode_labels,
    impute_plate_medians,
)
from idili_predict.normalize import normalize_all_plates

print("Imports OK")

## 2. Configuration

**Edit these paths to point to your own data.** The defaults use the included 63D example dataset.

In [ ]:
# ========== EDIT THESE PATHS ==========
INPUT_FILE = Path("../example_data/63D_well_median_fixed.csv")
DTYPES_FILE = Path("../example_data/column_dtypes.csv")
OUTPUT_DIR = Path("../results/walkthrough_63D")

# ========== METADATA COLUMN NAMES ==========
PLATE_COL = "Metadata_PlateID"
WELL_COL = "Metadata_WellID"
CMPD_COL = "Metadata_CMPD"
CONC_COL = "Metadata_CONC"
COND_COL = "Metadata_COND"

# ========== AUTOGLUON SETTINGS ==========
AG_TIME_LIMIT = 300    # seconds (increase for better models)
AG_PRESETS = "best_quality"
AG_NUM_BAG_FOLDS = 5

# ========== VALIDATE ==========
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Input:  {INPUT_FILE} (exists: {INPUT_FILE.exists()})")
print(f"Dtypes: {DTYPES_FILE} (exists: {DTYPES_FILE.exists()})")
print(f"Output: {OUTPUT_DIR}")

## 3. Load and Inspect Data

In [ ]:
# Load data
df = pd.read_csv(INPUT_FILE, low_memory=False)
print(f"Shape: {df.shape}")

# Remove duplicate columns (e.g., Metadata_Plate.1)
dup_cols = [c for c in df.columns if c.endswith(".1")]
if dup_cols:
    print(f"Removing {len(dup_cols)} duplicate columns")
    df = df.drop(columns=dup_cols)

# Validate required columns
validate_required_columns(df, [PLATE_COL, WELL_COL, CMPD_COL, CONC_COL, COND_COL])

print(f"\nCondition distribution:")
print(df[COND_COL].value_counts())
print(f"\nPlates: {df[PLATE_COL].nunique()}")
print(f"Compounds: {df[CMPD_COL].nunique()}")
print(f"Concentrations: {sorted(df[CONC_COL].dropna().unique())}")

In [ ]:
# Load column types
feature_cols_all, metadata_cols_all = load_column_types(DTYPES_FILE)
feature_cols = filter_to_present_numeric(df, feature_cols_all)
metadata_cols = [c for c in metadata_cols_all if c in df.columns]

print(f"Feature columns: {len(feature_cols)}")
print(f"Metadata columns: {len(metadata_cols)}")
print(f"\nFirst 5 features: {feature_cols[:5]}")

## 4. Encode Labels and Impute Missing Values

In [ ]:
# Encode labels: PC -> 1 (DILI positive), NC -> 0, DMSO -> 0
df = encode_labels(df, cond_col=COND_COL, cmpd_col=CMPD_COL)
print(f"\nLabel distribution:")
print(df["Metadata_Label_Encoded"].value_counts())

In [ ]:
# Impute missing values with per-plate medians
df = impute_plate_medians(df, feature_cols, plate_col=PLATE_COL)

## 5. Normalize Features (MAD Robustize)

Each plate is normalized independently using MAD robustize from pycytominer:
for each feature, `(x - median) / MAD` is computed across all wells on the plate.

In [ ]:
all_metadata = list(set(metadata_cols + ["Metadata_Label_Encoded"]))

df_normalized = normalize_all_plates(
    df, feature_cols, all_metadata, plate_col=PLATE_COL
)

print(f"\nNormalized shape: {df_normalized.shape}")
print(f"Feature mean: {df_normalized[feature_cols].mean().mean():.4f}")
print(f"Feature std:  {df_normalized[feature_cols].std().mean():.4f}")

## 6. Train AutoGluon

AutoGluon trains an ensemble of models using compound-grouped cross-validation. 
This prevents data leakage — the same compound at different doses never appears 
in both training and validation folds.

In [ ]:
from autogluon.tabular import TabularPredictor
from sklearn.model_selection import GroupKFold

# Prepare features
df_for_ag = df_normalized.copy()
df_for_ag["log_Metadata_CONC"] = np.log(df_for_ag[CONC_COL].replace(0, 0.01))

ag_feature_cols = feature_cols + ["log_Metadata_CONC"]
target_col = "Metadata_Label_Encoded"

# Filter to labeled rows
labeled_mask = df_for_ag[target_col].isin([0, 1])
train_df = df_for_ag.loc[labeled_mask].copy()

print(f"Training rows: {len(train_df)}")
print(f"Unique compounds: {train_df[CMPD_COL].nunique()}")
print(f"Classes: {train_df[target_col].value_counts().to_dict()}")

# Compound-grouped fold assignments
gkf = GroupKFold(n_splits=AG_NUM_BAG_FOLDS)
fold_assignments = np.zeros(len(train_df), dtype=int)
for fold_id, (_, val_idx) in enumerate(
    gkf.split(train_df, groups=train_df[CMPD_COL])
):
    fold_assignments[val_idx] = fold_id
train_df["AG_fold"] = fold_assignments

# Sample weights for class balancing
class_counts = train_df[target_col].value_counts()
n_samples = len(train_df)
n_classes = len(class_counts)
weight_map = {
    cls: n_samples / (n_classes * count)
    for cls, count in class_counts.items()
}
train_df["sample_weight"] = train_df[target_col].map(weight_map)

train_data = train_df[ag_feature_cols + [target_col, "AG_fold", "sample_weight"]]

print(f"\nFold distribution:")
for fold in range(AG_NUM_BAG_FOLDS):
    mask = train_df["AG_fold"] == fold
    n_cmpds = train_df.loc[mask, CMPD_COL].nunique()
    print(f"  Fold {fold}: {mask.sum()} wells, {n_cmpds} compounds")

In [ ]:
# Train AutoGluon
model_dir = OUTPUT_DIR / "autogluon_model"

predictor = TabularPredictor(
    label=target_col,
    eval_metric="roc_auc",
    problem_type="binary",
    path=str(model_dir),
    groups="AG_fold",
    sample_weight="sample_weight",
)

predictor.fit(
    train_data=train_data,
    presets=AG_PRESETS,
    time_limit=AG_TIME_LIMIT,
    verbosity=2,
    num_bag_folds=AG_NUM_BAG_FOLDS,
)

print(f"\nModel saved to: {model_dir}")

## 7. Evaluate OOF Predictions

In [ ]:
from sklearn.metrics import (
    roc_auc_score, accuracy_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay,
)
import matplotlib.pyplot as plt

# Get OOF predictions
oof_proba = predictor.predict_proba_oof()
oof_preds = predictor.predict_oof()

positive_col = 1 if 1 in oof_proba.columns else oof_proba.columns[-1]
prob_pos = oof_proba[positive_col]
y_true = train_data[target_col].loc[oof_proba.index]

# Metrics
print("OOF Metrics (compound-grouped cross-validation):")
print(f"  ROC AUC:       {roc_auc_score(y_true, prob_pos):.4f}")
print(f"  Accuracy:      {accuracy_score(y_true, oof_preds):.4f}")
print(f"  F1 (positive): {f1_score(y_true, oof_preds, pos_label=1):.4f}")
print(f"  F1 (negative): {f1_score(y_true, oof_preds, pos_label=0):.4f}")

# Leaderboard
leaderboard = predictor.leaderboard(silent=True)
print("\nModel Leaderboard:")
display(leaderboard.head(10))

In [ ]:
# Confusion matrix and ROC curve
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Confusion matrix
cm = confusion_matrix(y_true, oof_preds)
ConfusionMatrixDisplay(cm, display_labels=["NC (0)", "PC (1)"]).plot(
    ax=axes[0], cmap="Blues"
)
axes[0].set_title("OOF Confusion Matrix")

# ROC curve
RocCurveDisplay.from_predictions(y_true, prob_pos, ax=axes[1], name="OOF")
axes[1].plot([0, 1], [0, 1], "k--", alpha=0.3)
axes[1].set_title("OOF ROC Curve")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "oof_evaluation.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Figure saved to: {OUTPUT_DIR / 'oof_evaluation.png'}")

## 8. Export Results

In [ ]:
# Save OOF predictions
oof_df = train_df[
    [PLATE_COL, WELL_COL, CMPD_COL, CONC_COL, COND_COL, target_col]
].copy()
oof_df["Predicted_Prob_Positive"] = prob_pos.values
oof_df["Predicted_Label"] = (prob_pos.values >= 0.5).astype(int)
oof_df["Actual_Label"] = train_df[target_col].values

oof_file = OUTPUT_DIR / "oof_predictions.csv"
oof_df.to_csv(oof_file, index=False)

# Save leaderboard
leaderboard.to_csv(OUTPUT_DIR / "leaderboard.csv", index=False)

print(f"OOF predictions saved: {oof_file}")
print(f"Leaderboard saved: {OUTPUT_DIR / 'leaderboard.csv'}")
print(f"\nOutput files:")
for f in sorted(OUTPUT_DIR.glob("*")):
    if f.is_file():
        print(f"  {f.name}: {f.stat().st_size / 1e6:.1f} MB")